<h1 style="text-align: center;">Hiram</h1>

Hiram is a free financial library built in python that can be used for option pricing and management of financial instruments 

**Links**
- [Github repo](https://github.com/paulbqnt/hiram)
- [LinkedIn](https://www.linkedin.com/in/paulboquant/)
- [Youtube video](https://www.youtube.com/watch?v=he_yCcZ_YVU)

### **Table of Content**

1. [Option creation and Pricing](#section-one)
2. [Greeks](#section-two)
3. [Stock Creation](#section-three)
4. [Portfolio Creation](#section-four)
5. [Portfolio Greeks](#section-five)
6. [Delta hedging of the portfolio](#section-six)
7. [Market Maker Example](#section-seven)
8. [Delta hedging](#section-eight)
9. [Conclusion](#section-end)

<a id="section-one"></a>
### **1. Option creation and Pricing**


In [1]:
from option import BlackScholes, Way_option

example_call = BlackScholes(way=Way_option.call, S=100, K=105, T=1, r=0.05, sigma=0.3, )

example_call.dict() #Pydantic BaseModel function, makes the output easier to read

{'S': 100.0,
 'K': 105.0,
 'T': 1.0,
 'r': 0.05,
 'sigma': 0.3,
 'way': <Way_option.call: 'call'>,
 'stock': None,
 'quantity': 1,
 'price': 1,
 'id_': UUID('ad45fecd-e22c-4071-9e28-64799e7fbf78')}

<a id="section-two"></a>
### **2. Greeks**


#### Delta
${\displaystyle \Delta ={\frac {\partial V}{\partial S}}}$

Delta 0.53 or 53%. For 1€ rise (fall) in the spot price the value of the long call position will increase (decrease) by about 0.53€. **S** and the **option value** move in the same direction.

In [2]:
example_call.delta()

0.5338376178017635

1. [Introduction](#intro)<br>

#### Gamma
${\displaystyle \Gamma ={\frac {\partial \Delta }{\partial S}}={\frac {\partial ^{2}V}{\partial S^{2}}}}$

Gamma 0.01 or 1%. For 1€ rise (fall) in the spot price the value of the delta will increase (decrease) by about 0.01€.

In [3]:
example_call.gamma()

0.013141252320879648

#### Vega
${\displaystyle {\mathcal {V}}={\frac {\partial V}{\partial \sigma }}}$

Vega 0.39, if volatility rises (fall) by 1%, the value of the long call will increase (decrease) by about 0.39€.

In [4]:
example_call.vega()

0.3942375696263895

#### Rho
${\displaystyle \rho ={\frac {\partial V}{\partial r}}}$

Rho 0.44. If interest rates rise (fall) by 1%, the value of the long call position will increase (decrease) by about 0.44€.

In [5]:
example_call.rho()

0.44143924313138455

#### Theta
${\displaystyle \Theta =-{\frac {\partial V}{\partial \tau }}}$

Theta -0.22. Each day, the value of the option will fall by about 0.22€. Negative relationship

In [6]:
example_call.theta()

-0.02224865687685689

#### Compute all

In [7]:
example_call.compute_all()

'value: 11.976881462184025, delta: 0.5338376178017635, gamma: 0.013141252320879648, vega: 0.3942375696263895, rho: 0.44143924313138455, theta: -0.02224865687685689'

<a id="section-three"></a>
### **3. Stock Creation**


In [8]:
from stock import Stock, Way_stock

example_stock = Stock(ticker="AAPL")

print(f"stock ticker:\t{example_stock.ticker}")
print(f"stock price:\t{example_stock.price}")
print(f"stock way:\t{example_stock.way}")
example_stock.hist.tail()

stock ticker:	AAPL
stock price:	180.08999633789062
stock way:	Way_stock.long


open        high         low       close    volume  \
symbol date                                                                   
AAPL   2023-05-25  172.410004  173.899994  171.690002  172.990005  56058300   
       2023-05-26  173.320007  175.770004  173.110001  175.429993  54835000   
       2023-05-30  176.960007  178.990005  176.570007  177.300003  55964400   
       2023-05-31  177.330002  179.350006  176.759995  177.250000  99625300   
       2023-06-01  177.699997  180.119995  176.929993  180.089996  68813900   

                     adjclose  dividends  splits  
symbol date                                       
AAPL   2023-05-25  172.990005        0.0     0.0  
       2023-05-26  175.429993        0.0     0.0  
       2023-05-30  177.300003        0.0     0.0  
       2023-05-31  177.250000        0.0     0.0  
       2023-06-01  180.089996        0.0     0.0

#### Linkage to the call option

In [9]:
example_call.stock = example_stock

In [10]:
# Let's check
example_call.stock.ticker

'AAPL'

In [11]:
# Modification of S/K
example_call.S = example_stock.price
example_call.K = example_stock.price * 1.25

print(f"Spot:\t{example_call.S}")
print(f"Strike:\t{example_call.K}")

Spot:	180.08999633789062
Strike:	225.11249542236328


In [12]:
# compute all the greeks
example_call.compute_all()

'value: 10.249865700985907, delta: 0.3183163677151672, gamma: 0.006740312884853831, vega: 0.6558137079380103, rho: 0.500148738240813, theta: -0.03380260085006635'

<a id="section-four"></a>
### **4. Portfolio Creation**


In [13]:
from portfolio import Portfolio
example_port = Portfolio()
example_port.dict()

{'name': None,
 'underlyings': {'stocks': {},
  'options': {},
  'bonds': {},
  'portfolios': {},
  'hedge': {}},
 'id_': UUID('60e64398-b5c2-47d1-8a94-4ec1e7053fa3')}

### Portfolio functions

There's no option within our portfolio.

In [14]:
example_port.get_option_book_value()

0

Let's add the call

In [15]:
example_port.add_underlying(example_call)

In [16]:
# Now we have one option in the portfolio
len(example_port.underlyings["options"])

1

In [17]:
# compare the ID of the option within the port with example_call
print(example_port.underlyings["options"].keys())
print(example_call.id_)

dict_keys([UUID('ad45fecd-e22c-4071-9e28-64799e7fbf78')])
ad45fecd-e22c-4071-9e28-64799e7fbf78


<a id="section-five"></a>
#### **5. Portfolio Greeks**

In [18]:
print(f"delta:\t {example_port.get_delta()}")
print(f"gamma:\t {example_port.get_gamma()}")
print(f"vega:\t {example_port.get_vega()}")
print(f"rho:\t {example_port.get_rho()}")
print(f"theta:\t {example_port.get_theta()}")
print(f"\ngreeks of the option: {example_call.compute_all()}")

delta:	 {'AAPL': 0.3183163677151672}
gamma:	 0.006740312884853831
vega:	 0.6558137079380103
rho:	 0.500148738240813
theta:	 -0.03380260085006635

greeks of the option: value: 10.249865700985907, delta: 0.3183163677151672, gamma: 0.006740312884853831, vega: 0.6558137079380103, rho: 0.500148738240813, theta: -0.03380260085006635


<a id="section-six"></a>
### **6. Delta hedging of the portfolio**
To delta hedge we'll use underlyings stocks, on which the vanilla options are written.

In [19]:
# this function will loop through the portfolio and add stocks into the hedge part
example_port.delta_hedge()

-0.0017675405307795765 AAPL added into the hedging part


In [20]:
# check if a stock has been added into hedge dict
example_port.underlyings["hedge"]

{UUID('1d168647-1cf1-400d-b64e-4f6f6f21ca4d'): Stock(way=<Way_stock.long: 'long'>, ticker='AAPL', price=180.08999633789062, quantity=-0.0017675405307795765, id_=UUID('1d168647-1cf1-400d-b64e-4f6f6f21ca4d'), hist=                         open        high         low       close     volume  \
 symbol date                                                                    
 AAPL   2018-06-04   47.910000   48.355000   47.837502   47.957500  105064800   
        2018-06-05   48.267502   48.485001   48.090000   48.327499   86264000   
        2018-06-06   48.407501   48.520000   47.980000   48.494999   83734400   
        2018-06-07   48.535000   48.549999   48.084999   48.365002   85388800   
        2018-06-08   47.792500   48.000000   47.442501   47.924999  106627200   
 ...                       ...         ...         ...         ...        ...   
        2023-05-25  172.410004  173.899994  171.690002  172.990005   56058300   
        2023-05-26  173.320007  175.770004  173.110001  175.

In [21]:
# We can also print the delta
example_port.get_delta()

{'AAPL': 0.0}

<a id="section-seven"></a>
### **7. Market Maker Example**

Here's several stocks from the french index CAC40

In [22]:
example_tickers = ["GLE.PA", "BNP.PA", "MC.PA", "CDI.PA"]
stocks_list = []

for ticker in example_tickers:
    temp_stock = Stock(ticker=ticker)
    stocks_list.append(temp_stock)

In [23]:
for stock in stocks_list:
    print(stock.hist.tail(5))

                        open       high        low      close      volume  \
symbol date                                                                 
GLE.PA 2023-05-25  23.879999  24.035000  23.340000  23.840000   3669032.0   
       2023-05-26  23.905001  23.980000  23.290001  23.840000   3272092.0   
       2023-05-29  24.200001  24.225000  23.455000  23.700001   3066398.0   
       2023-05-30  22.075001  22.565001  21.910000  22.309999   7660579.0   
       2023-05-31  22.084999  22.145000  21.590000  21.629999  13408871.0   

                    adjclose  dividends  
symbol date                              
GLE.PA 2023-05-25  22.129957        0.0  
       2023-05-26  22.129957        0.0  
       2023-05-29  22.000000        0.0  
       2023-05-30  22.309999        1.7  
       2023-05-31  21.629999        0.0  
                        open       high        low      close     volume  \
symbol date                                                                
BNP.PA 2023-05

In [24]:
# import random functions
from random import choice, uniform

In [25]:
ways = [Way_option.call, Way_option.put]
example_port2 = Portfolio()

# let's write and add 100 options in our portfolio
for i in range(0,100):
    temp_stock = choice(stocks_list)
    temp_option = BlackScholes(way=choice(ways),
                      S = uniform(temp_stock.price * 0.5,temp_stock.price * 1.5),
                      K = uniform(temp_stock.price * 0.5,temp_stock.price * 1.5),
                      T = uniform(0.02, 3.0),
                      r = uniform(0.01, 0.15),
                      sigma = uniform(0.1, 0.75),
                      quantity = uniform(1, 2000),
                      price = uniform(10, 150),
                     stock = temp_stock)
    example_port2.add_underlying(temp_option)

we have now our list of stocks. Let's write 100 vanilla calls/puts randomly,
and add these options into a new portfolio

#### Portfolio informations

In [26]:
print(f"Portfolio option book value:\t{example_port2.get_option_book_value()}")
print(f"Portfolio delta:\t{example_port2.get_delta()}")

Portfolio option book value:	8559502.004710117
Portfolio delta:	{'MC.PA': 367904.94572213973, 'GLE.PA': 473878.5910277292, 'CDI.PA': 486671.67686180724, 'BNP.PA': 637097.4683161988}


<a id="section-height"></a>
### **8.Delta hedging**


In [27]:
example_port2.delta_hedge()

-452.0271958303471 MC.PA added into the hedging part
-21908.39618187642 GLE.PA added into the hedging part
-629.1812241264478 CDI.PA added into the hedging part
-11795.917202652367 BNP.PA added into the hedging part


In [28]:
example_port2.get_delta()

{'MC.PA': 0.0,
 'GLE.PA': 5.820766091346741e-11,
 'CDI.PA': -1.1641532182693481e-10,
 'BNP.PA': -1.1641532182693481e-10}

####

<a id="section-end"></a>
Well done, portfolio is now delta hedged !


![](https://media2.giphy.com/media/v1uV0oxObr9ZT48Kpa/200w.gif)